# 🧱 Mini LLM：从零构建一个能「说话」的模型

## 这个 Notebook 的目标

学完后，你能：
1. **随便看懂任何 tokenizer 的内部原理**（BPE、WordPiece、SentencePiece…万变不离其宗）
2. **完全理解 LLM 训练时 loss 到底在算什么**（token 级别还是句子级别？一目了然）
3. **从零手写一个小 GPT**，并且真的能训练、能生成

## 前置知识

- 会写 Python
- 知道什么是矩阵乘法（线性代数高一水平就够了）
- 用过 PyTorch 的基本语法（`torch.tensor`、`nn.Module`）

## 我们不会用

- ❌ huggingface transformers
- ❌ tiktoken / sentencepiece
- ❌ 任何现成的 tokenizer 库

一切从 Python 标准库 + PyTorch 手写。

---

## 第一部分：Tokenizer — LLM 的「翻译官」

大问题：**计算机只认识 0 和 1，不认识文字。那 LLM 是怎么「读懂」英文/中文的？**

答案：有一个东西把文字翻译成数字，LLM 处理这些数字，再翻译回来。

这个东西就叫 **Tokenizer**。

下面我们从一个具体问题出发，一步步走到工业级方案。

### 0. 准备我们的语料库

Tokenizer 需要从文本中「学习」切分规则，所以先准备一小段英文语料。

**这里选英文是因为 BPE 最初就是为英文设计的（byte 级别），逻辑最直观。** 等学完原理，中文处理就是一层窗户纸。

In [1]:
# 我们的语料库 — 故意选有重复模式的简单句子
# 这样 BPE 合并时能清楚看到高频 pair 如何被合并
corpus = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "the cat and the dog",
    "i love my cat",
    "i love my dog",
    "the cat is cute",
    "the dog is happy",
    "the mat is soft",
    "the log is hard",
    "cats and dogs are friends",
]

print(f"语料库共 {len(corpus)} 条句子")
for i, s in enumerate(corpus):
    print(f"  [{i}] {s}")

total_chars = sum(len(s) for s in corpus)
print(f"\n总字符数: {total_chars}")
print(f"总词数（按空格算）: {sum(len(s.split()) for s in corpus)}")

语料库共 10 条句子
  [0] the cat sat on the mat
  [1] the dog sat on the log
  [2] the cat and the dog
  [3] i love my cat
  [4] i love my dog
  [5] the cat is cute
  [6] the dog is happy
  [7] the mat is soft
  [8] the log is hard
  [9] cats and dogs are friends

总字符数: 175
总词数（按空格算）: 46


### 1. 第一个问题：计算机怎么「看懂」文字？

想象你是计算机。在键盘上敲了 `the cat`，计算机内部只看到一堆数字——因为所有字符在计算机里都有唯一编号（ASCII/Unicode）。

```
文本:    t    h    e         c    a    t
        ↓    ↓    ↓    ↓    ↓    ↓    ↓
ASCII:  116  104  101  32  99   97   116
```

**所以 Tokenizer 的本质是一个映射表：**

```
文本  ──编码(encode)──→  数字序列  ──模型处理──→  数字序列  ──解码(decode)──→  文本
```

核心问题是：**切多细？**

- 方案一：按**字符**切 → 每个字母一个 token
- 方案二：按**词**切 → 每个单词一个 token  
- 方案三：按**子词**切 → 把单词拆成更小的有意义的片段（这就是 BPE）

下面我们一个个试。

### 2. 方案一：字符级 Tokenizer（最朴素的想法）

**思路：把文本拆碎，每个字符都是一个 token。**

```
"the cat"
  ↓ 按字符切分
['t', 'h', 'e', ' ', 'c', 'a', 't']
  ↓ 查表给编号（词表里每个字符一个 ID）
[5, 12, 3, 0, 6, 1, 8]
```

**词表（vocab）**就是一张「字符 → ID」的字典。

我们先手写一个最简单的实现，看看效果。

In [2]:
class CharTokenizer:
    """
    字符级 Tokenizer：一个字符 = 一个 token
    只有两个核心方法：encode（编码）和 decode（解码）
    """
    
    def __init__(self):
        # stoi = string-to-id，把字符映射到数字
        # itos = id-to-string，把数字映射回字符
        self.stoi = {}
        self.itos = {}
    
    def train(self, texts):
        """
        训练：从语料中收集所有出现过的字符，建立词表
        
        步骤：
        1. 把所有文本拼成一个长字符串
        2. 用 set 去重 → 得到所有字符
        3. 排序 → 给每个字符分配一个唯一 ID
        """
        all_text = "".join(texts)
        chars = sorted(list(set(all_text)))
        
        self.stoi = {ch: i for i, ch in enumerate(chars)}
        self.itos = {i: ch for i, ch in enumerate(chars)}
        
        print(f"=== 字符级 Tokenizer 训练完成 ===")
        print(f"词表大小: {len(self.stoi)} 个 token")
        print(f"词表内容: {chars}")
    
    def encode(self, text):
        """编码：文本 → token ID 列表"""
        return [self.stoi[ch] for ch in text]
    
    def decode(self, ids):
        """解码：token ID 列表 → 文本"""
        return "".join([self.itos[i] for i in ids])


In [3]:
# 训练并测试
char_tokenizer = CharTokenizer()
char_tokenizer.train(corpus)

# 拿一句来试试
text = "the cat"
ids = char_tokenizer.encode(text)
recovered = char_tokenizer.decode(ids)

print(f"\n=== 编码/解码测试 ===")
print(f"原文:     '{text}'")
print(f"Token IDs: {ids}")
print(f"解码回来: '{recovered}'")
print(f"\n关键观察：原文 {len(text)} 个字符 → 产生 {len(ids)} 个 token → 一样多！")
print(f"每个字符都变成一个 token，没有压缩。")

=== 字符级 Tokenizer 训练完成 ===
词表大小: 20 个 token
词表内容: [' ', 'a', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'v', 'y']

=== 编码/解码测试 ===
原文:     'the cat'
Token IDs: [16, 7, 4, 0, 2, 1, 16]
解码回来: 'the cat'

关键观察：原文 7 个字符 → 产生 7 个 token → 一样多！
每个字符都变成一个 token，没有压缩。


### 3. 字符级 Tokenizer 的致命弱点

字符级很好理解，但有两个大问题：

**问题 1：序列太长。**
- `"the cat"` = 7 个字符 → 7 个 token
- 一篇 1000 字符的文章 → 1000 个 token
- LLM 里有个叫 Self-Attention 的东西，它的计算量和序列长度是**平方关系**：1000 个 token 比 100 个 token 慢 100 倍，不是 10 倍！

**问题 2：语义被打散。**
- `"cat"` 是一个完整的意思（=🐱）
- 但被拆成了 `'c', 'a', 't'` 三个互不相关的 token
- 模型需要先「自己学会」把 `c+a+t` 拼回 cat 的概念
- 白白浪费了模型的算力去做这件本可以预处理好的事

那自然而然会想：按词切不就完了？

### 4. 方案二：词级 Tokenizer

**思路：按空格切分，每个单词是一个 token。**

```
"the cat sat on the mat"
  ↓ 按空格切
['the', 'cat', 'sat', 'on', 'the', 'mat']
  ↓ 查表
[0, 1, 2, 3, 0, 4]
```

**优点**：`"the cat"` 只要 2 个 token！序列短多了。

我们也手写一个看看。

In [ ]:
class WordTokenizer:
    """
    词级 Tokenizer：按空格切分，一个词 = 一个 token
    结构和 CharTokenizer 一模一样，只是切分粒度从「字符」变成了「词」
    """
    
    def __init__(self):
        self.stoi = {}
        self.itos = {}
    
    def train(self, texts):
        """从语料中收集所有出现过的词"""
        all_words = set()
        for text in texts:
            words = text.split()
            all_words.update(words)
        
        all_words = sorted(list(all_words))
        self.stoi = {w: i for i, w in enumerate(all_words)}
        self.itos = {i: w for i, w in enumerate(all_words)}
        
        print(f"=== 词级 Tokenizer 训练完成 ===")
        print(f"词表大小: {len(self.stoi)} 个词")
        print(f"词表内容: {all_words}")
    
    def encode(self, text):
        """把文本按空格切开，每个词查表"""
        return [self.stoi[w] for w in text.split()]
    
    def decode(self, ids):
        """把 ID 查表变回词，用空格拼起来"""
        return " ".join([self.itos[i] for i in ids])


In [ ]:
# 训练并测试
word_tokenizer = WordTokenizer()
word_tokenizer.train(corpus)

text = "the cat sat on the mat"
ids = word_tokenizer.encode(text)
recovered = word_tokenizer.decode(ids)

print(f"\n=== 编码/解码测试 ===")
print(f"原文:     '{text}'")
print(f"Token IDs: {ids}")
print(f"解码回来: '{recovered}'")
print(f"\n关键观察：原文 {len(text.split())} 个词 → 产生 {len(ids)} 个 token → 序列比字符级短多了！")

In [ ]:
# 但词级 tokenizer 有一个致命问题：遇到生词（OOV）怎么办？
print("=== OOV（Out Of Vocabulary）问题演示 ===")
print(f"当前词表: {list(word_tokenizer.stoi.keys())}")
print()

try:
    test_text = "the elephant sat"
    print(f"尝试编码: '{test_text}'")
    ids = word_tokenizer.encode(test_text)
    print(f"结果: {ids}")
except KeyError as e:
    # 看看出错时发生了什么
    words = test_text.split()
    for w in words:
        if w in word_tokenizer.stoi:
            print(f"  ✅ '{w}' → ID {word_tokenizer.stoi[w]}  (词表里有)")
        else:
            print(f"  ❌ '{w}' → 不在词表中！报错！")
    print(f"\n结论：遇到生词 'elephant'，整个程序崩溃。")
    print(f"而在现实世界，你永远不可能预先把所有词都收进词表。")

### 5. 三种方案的对比

现在我们有三个方案：只认识字符、只认识词，以及一个还没学的折中方案。

| 方案 | 词表大小 | 序列长度 | OOV 问题 | 语义密度 |
|------|---------|---------|---------|---------|
| 字符级 | 很小（~100） | ❌ 很长 | ✅ 无 | ❌ 太碎 |
| 词级 | ❌ 巨大（10万+） | ✅ 短 | ❌ 会崩溃 | ✅ 完整 |
| **子词级** | ✅ 可控 | ✅ 中等 | ✅ 极少 | ✅ 较好 |

**我们需要的是第三种：子词级 (Subword)。**

理想效果：
```
"the cat sat"     → [the, cat, sat]      ← 常见词整体保留
"cats and dogs"   → [cat, s, and, dog, s]  ← 变体拆成词根+后缀
"unbelievable"    → [un, believ, able]     ← 生僻词拆成有意义的片段
```

**核心思想**：
- 高频出现的字符组合 → 合并成新 token，整体保留
- 低频长尾的单词 → 拆成更小的片段
- 词表大小可控（想多大调多大）

这就是 **BPE（Byte Pair Encoding）** 做的事情。

### 6. BPE 预告：合并的直觉

BPE 的核心操作就一句话：**找出现最频繁的相邻字符对，把它们合并成一个新 token。**

用我们的语料举个例子：

```
初始：每个字符一个 token
['t', 'h', 'e', ' ', 'c', 'a', 't', ...]
               ↓ 统计相邻 pair 频率
('t', 'h') 出现了 20 次 ← 最高频！
               ↓ 合并
新 token: 'th'
               ↓ 把语料里所有 't'+'h' 都换成 'th'
['th', 'e', ' ', 'c', 'a', 't', ...]
               ↓ 继续统计，继续合并...
```

重复这个操作 N 次，就能得到你想要的任何大小的词表。

**下一 Part**：从零手写完整的 BPE 训练、编码、解码。每一步合并都会打印出来给你看，让你看到词表如何在眼前「生长」。

---

### 第一部分小结

在进入 BPE 实现之前，确认你已经理解了这些：

1. ✅ Tokenizer = 文本 ↔ 数字的翻译器
2. ✅ encode = 文本→ID序列，decode = ID序列→文本
3. ✅ 词表（vocab）= 所有合法 token 的集合，每个 token 有唯一 ID
4. ✅ 字符级：简单但序列太长、语义太碎
5. ✅ 词级：序列短但词表爆炸、遇生词崩溃（OOV）
6. ✅ 子词级：折中方案，高频组合合并成 token，低频词拆成片段

**如果以上 6 点有任何不清楚的，回头重看对应部分。**

全部清楚了？→ 下一步，我们动手写 BPE。